# 14 — Explore and select a 12-month reporting window

**Project:** Clinical Microbiology Analytics Lakehouse  
**Source:** ARMD release `2025-04-11`  
**Warehouse:** `workspace.micro_dev`

This notebook records the availability checks that supported selecting **January–December 2023 in shifted dates**. The saved outputs include a year-level patient–organism summary and a monthly distinct-culture check. Final reporting coverage is measured after category-specific first-culture selection in Notebook 13.

## 1. Availability by shifted year

The query reads the full-release `int_reporting_culture_organism` model, which is upstream of the optional reporting-window filter. It remains suitable for exploring candidate years after a period-specific mart has been built.

- `patient_organism_groups` counts distinct patient–organism groups within each shifted year/cohort; a patient can occur in multiple organisms or years.
- The phenotype-evidence columns count groups with at least one positive observation during that shifted year.
- These are preselection availability counts, not category-specific first-culture counts or antibiotic-specific S/I/R denominators.
- The saved output is limited to the first 20 rows, so it is not a complete inventory of every shifted year.

For 2023, the saved year summary records 922 blood, 715 respiratory, and 7,528 urine patient–organism groups (9,165 total), with 790 groups showing ESCE-R evidence, 115 MRSA evidence, and 28 VRE evidence. Those VRE availability counts cannot yield 30 categorized results in any single species/cohort/drug subgroup under the same scope. The final VRE display outcome is checked directly in Notebook 13.

Patient-level date shifts prevent interpreting this as a shared actual clinical calendar year. Use the label **2023 shifted-date research antibiogram**.

In [1]:
%sql
with distinct_culture_organisms as (
    select distinct
        source_version, cohort_id, patient_key, organism_reporting_group,
        culture_key, cast(culture_time_shifted as date) as shifted_date,
        ast_esce_enterobacterales_status,
        ast_oxacillin_resistance_status,
        ast_vancomycin_resistance_status
    from workspace.micro_dev.int_reporting_culture_organism
    where culture_time_shifted is not null
),
patient_year as (
    select year(shifted_date) as shifted_year, cohort_id,
           organism_reporting_group, patient_key,
           max(case when ast_esce_enterobacterales_status = 'positive'
                     then 1 else 0 end) as any_esce_positive,
           max(case when ast_oxacillin_resistance_status = 'positive'
                     then 1 else 0 end) as any_mrsa_positive,
           max(case when ast_vancomycin_resistance_status = 'positive'
                     then 1 else 0 end) as any_vre_positive
    from distinct_culture_organisms
    group by year(shifted_date), cohort_id, organism_reporting_group, patient_key
)
select shifted_year, cohort_id,
       count(*) as patient_organism_groups,
       sum(any_esce_positive) as groups_with_esce_evidence,
       sum(any_mrsa_positive) as groups_with_mrsa_evidence,
       sum(any_vre_positive) as groups_with_vre_evidence
from patient_year
group by shifted_year, cohort_id
order by shifted_year desc, cohort_id;

,shifted_year,cohort_id,patient_organism_groups,groups_with_esce_evidence,groups_with_mrsa_evidence,groups_with_vre_evidence
0,2024,blood,22,3,1,0
1,2024,respiratory,24,1,1,0
2,2024,urine,183,15,0,0
3,2023,blood,922,86,59,12
4,2023,respiratory,715,21,34,0
5,2023,urine,7528,683,22,16
6,2022,blood,961,74,55,11
7,2022,respiratory,698,20,48,0
8,2022,urine,7273,558,18,26
9,2021,blood,907,72,59,5


## 2. Monthly distribution within 2023

This query counts **distinct culture keys** in each shifted month/cohort from the reporting-organism model before final first-culture selection. It is restricted explicitly to 2023, independent of the active mart variable. The counts describe cultures represented in this reviewed reporting layer, not every raw culture order or unique patients. Positive counts in each month support evaluating the period; final drug-specific display coverage still requires Notebook 13.

In [2]:
%sql
select
    date_format(culture_time_shifted, 'yyyy-MM')
        as shifted_month,

    count(distinct case
        when cohort_id = 'blood'
        then culture_key
    end) as blood_cultures,

    count(distinct case
        when cohort_id = 'respiratory'
        then culture_key
    end) as respiratory_cultures,

    count(distinct case
        when cohort_id = 'urine'
        then culture_key
    end) as urine_cultures

from workspace.micro_dev.int_reporting_culture_organism

where culture_time_shifted >= timestamp '2023-01-01 00:00:00'
  and culture_time_shifted < timestamp '2024-01-01 00:00:00'

group by date_format(culture_time_shifted, 'yyyy-MM')
order by shifted_month;

,shifted_month,blood_cultures,respiratory_cultures,urine_cultures
0,2023-01,82,69,686
1,2023-02,91,57,621
2,2023-03,79,53,691
3,2023-04,73,62,706
4,2023-05,75,52,721
5,2023-06,90,66,716
6,2023-07,84,64,783
7,2023-08,77,60,812
8,2023-09,68,70,804
9,2023-10,91,72,774


## 3. Recorded monthly result and window decision

Every shifted month from January through December 2023 contains culture records in all three cohorts.

| Cohort | Sum of monthly culture counts | Monthly range |
|---|---:|---:|
| Blood | 961 | 68–91 |
| Respiratory | 719 | 44–72 |
| Urine | 8,611 | 546–812 |

December has the lowest respiratory and urine counts. The table establishes representation across the year; it does not prove complete source capture or explain the lower counts. The culture totals differ from the year-level patient–organism counts because the queries measure different grains.

The configured window is **2023-01-01 inclusive to 2024-01-01 exclusive**. `analysis_window_start` is set before dbt builds the selected populations; filtering a finished full-release mart would produce a different result. The saved full build passed 224 tests, and Notebook 13's dated coverage result confirms **424 mart rows, 194 displayable, 230 insufficient data** for this period.

The narrower window has substantially less panel coverage than the full-release baseline (381 displayable rows). Retain the 30-result threshold and preserve full-release outputs separately for comparison. No VRE susceptibility percentages meet the threshold in 2023; this is a small-population limitation, not evidence of no VRE.

For future window changes, rebuild the marts, rerun current-scope validation, and record the new dates and coverage before refreshing exports. These are descriptive research outputs based on shifted dates and source-reported categories. The existing SQL, execution counts, and outputs were preserved in this documentation revision.